<a href="https://colab.research.google.com/github/rhonaeloisa/bigdata_lab4/blob/lab4/big_data_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import format_number


spark = SparkSession.builder.appName('lab4').getOrCreate()


In [ ]:
#create schema for flood control csv
df_schema = StructType([
    StructField("MainIsland", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Province", StringType(), True),
    StructField("LegislativeDistrict", StringType(), True),
    StructField("Municipality", StringType(), True),
    StructField("DistrictEngineeringOffice", StringType(), True),
    StructField("ProjectID", StringType(), True),
    StructField("ProjectName", StringType(), True),
    StructField("TypeOfWork", StringType(), True),
    StructField("FundingYear", IntegerType(), True),
    StructField("ContractId", StringType(), True),
    StructField("ApprovedBudgetForContract", IntegerType(), True),
    StructField("ContractCost", IntegerType(), True),
    StructField("ActualCompletionDate", StringType(), True),
    StructField("Contractor", StringType(), True),
    StructField("ContractorCount", StringType(), True),
    StructField("StartDate", StringType(), True)
])

#Load CSV as a DataFrame
df = spark.read.csv("/content/dpwh_flood_control_projects_lab4.csv", header=True, schema=df_schema)

#data cleaning
#remove duplicate rows
duplicate_rows = (
    df.groupBy(df.columns)
      .count()
      .filter(col("count") > 1)
      .select("ProjectName", "count")
)

print("Duplicate rows in CSV file")
duplicate_rows.show(truncate=False)

cleaned_data = df.dropDuplicates()

check_duplicates = (
    cleaned_data.groupBy(cleaned_data.columns)
      .count()
      .filter(col("count") > 1)
      .select("ProjectName", "count")
)

print("Check Duplicate rows in CSV file after cleaning")
check_duplicates.show()

Duplicate rows in CSV file
+--------------------------------------------------------------------------------------------------------------------------+-----+
|ProjectName                                                                                                               |count|
+--------------------------------------------------------------------------------------------------------------------------+-----+
|Construction of Shore Protection Structure along Poblacion 2, Burgos, Surigao del Norte                                   |5    |
|Reconstruction/Rehabilitation of Seawall and Reclamation in Barangay Sta. Cruz, Municipality of Socorro, Surigao del Norte|3    |
|Construction of Shore Protection,Brgy. Pamosaingan,Socorro, Surigao del Norte                                             |3    |
|Construction of Slope Protection Structure along Jct. San Benito-Sta. Monica Road, San Benito, Surigao Del Norte          |3    |
|Construction of Flood Control Structure, Nuevo Campo, S

In [ ]:
#INSIGHT #1
#Show the top 5 contractors by number of projects handled

#Split the Contractor by '/'
#to separate multiple contractor listed per row
contractor_split = cleaned_data.withColumn(
    "Contractor",
    explode(split(col("Contractor"), "/"))
)

#remove spaces so that all with the same contractor name will be treated the same
contractor_split = contractor_split.withColumn(
    "Contractor",
    trim(col("Contractor"))
)

# Count the total projects per contractor
#limit only to 5 to show top 5
#show in descending order
top_contractors = (contractor_split.groupBy("Contractor")
                    .agg(count("*").alias("NumOfProj"))
                    .orderBy(col("NumOfProj")
                    .desc())
                    .limit(5)
)
print("Top 5 contractors by number of projects")
top_contractors.show(truncate=False)

Top 5 contractors by number of projects
+---------------------------------------------------------------+---------+
|Contractor                                                     |NumOfProj|
+---------------------------------------------------------------+---------+
|LEGACY CONSTRUCTION CORPORATION (FORMERLY: LEGACY CONSTRUCTION)|133      |
|ALPHA & OMEGA GEN. CONTRACTOR & DEVELOPMENT CORP.              |105      |
|ST. TIMOTHY CONSTRUCTION CORPORATION                           |105      |
|QM BUILDERS                                                    |96       |
|EGB CONSTRUCTION CORPORATION (FORMERLY: EGB CONSTRUCTION)      |95       |
+---------------------------------------------------------------+---------+



In [ ]:
#INSIGHT #2
#Show the total budget per province of region V

#filter region 5 first
regionV = cleaned_data.filter(col("Region") == "Region V")

#then group the province within region 5 and calculate the total budget
budget_per_province = (regionV.groupBy("Province")
                      .agg(sum("ContractCost")
                      .alias("TotalBudget"))
                      .orderBy(col("TotalBudget")
                      .desc())
)

print("Total Budget per Province in Region V(2018-2025)")
budget_per_province_formatted = budget_per_province.withColumn(
    "TotalBudget", format_number("TotalBudget", 0)
)

budget_per_province_formatted.show(truncate=False)

Total Budget per Province in Region V(2018-2025)
+---------------+-------------+
|Province       |TotalBudget  |
+---------------+-------------+
|Camarines Sur  |7,861,296,417|
|Albay          |6,877,128,607|
|Sorsogon       |1,085,104,537|
|Masbate        |920,525,160  |
|Catanduanes    |455,003,380  |
|Camarines Norte|48,250,000   |
+---------------+-------------+



In [ ]:
#INSIGHT #3
#Total Budget per year

budget_per_year = (
    cleaned_data.filter(col("FundingYear").isNotNull())
                        .groupBy("FundingYear")
                        .agg(sum("ContractCost").alias("TotalBudget"))
                        .orderBy(col("FundingYear").desc())
)

budget_per_year_formatted = budget_per_year.withColumn(
    "TotalBudget", format_number("TotalBudget", 0)
)

print("Total Budget per Year")
budget_per_year_formatted.show(truncate=False)

Total Budget per Year
+-----------+---------------+
|FundingYear|TotalBudget    |
+-----------+---------------+
|2025       |1,450,632,964  |
|2024       |144,628,793,326|
|2023       |203,376,105,562|
|2022       |163,642,310,423|
|2021       |28,313,726,674 |
|2020       |2,281,209,747  |
|2019       |335,826,929    |
|2018       |734,761,555    |
+-----------+---------------+



In [ ]:
#parquet file
parquet_path = "/content/parquet"
cleaned_data.write.mode("overwrite").format("parquet").save(parquet_path)
